# Agent with Code Interpreter

This notebook creates a Foundry agent with the **Code Interpreter** tool enabled using the
[`azure-ai-projects`](https://pypi.org/project/azure-ai-projects/) SDK.

**What this notebook does:**
1. Authenticates to Azure using `DefaultAzureCredential`
2. Uploads a CSV file of synthetic company quarterly financial results to the Foundry Files API for the agent to analyse
3. Creates a versioned agent with `CodeInterpreterTool` attached to the uploaded file
4. Sends a prompt asking the agent to generate a bar chart for a specific sector
5. Extracts the `container_file_citation` annotation from the response
6. Downloads the generated PNG chart to the local directory

> **Code Interpreter** runs Python in a sandboxed Azure container. Each conversation creates
> its own session, active for up to one hour (30-minute idle timeout). Additional charges
> apply beyond token-based fees — see the
> [pricing page](https://azure.microsoft.com/pricing/details/cognitive-services/openai-service/).

## Prerequisites

1. **Python environment**: Run `uv sync` from the repository root to create the
   shared `.venv`, then select the `.venv` kernel in VS Code.
2. **`.env` file**: Must be populated by the `04-foundry-project-pattern-setup` labs:
   - `ALPHA_FOUNDRY_PROJECT_ENDPOINT` — Team Alpha project endpoint URL (set by Lab 1B)
   - `ALPHA_FOUNDRY_CORE_CONNECTION` — Team Alpha APIM connection name, e.g. `core-alpha` (set by Lab 1B)
   - `CHAT_MODEL` — chat model deployment name, e.g. `gpt-4.1-mini` (set by Lab 1A)
3. **Azure CLI**: Run `az login` before executing the cells.
4. **Permissions**: Your identity needs **Azure AI Developer** role on the Foundry project.
5. **CSV asset**: The file `assets/synthetic_500_quarterly_results.csv` is included
   in this repository and used as the input data file.

## 1. Imports and configuration

Load `.env` from the repository root and read the required environment variables.

In [1]:
import os
import subprocess
from pathlib import Path
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    AutoCodeInterpreterToolParam,
    CodeInterpreterTool,
    PromptAgentDefinition,
)

# ── Change this to rename your agent ─────────────────────────────────────────
AGENT_NAME = "code-interpreter-agent"
# ─────────────────────────────────────────────────────────────────────────────

repo_root = Path(subprocess.run(
    'git rev-parse --show-toplevel', shell=True, capture_output=True, text=True
).stdout.strip())
load_dotenv(repo_root / '.env', override=True)

# Team Alpha (1:1 spoke) — project endpoint and APIM connection (set by Labs 1A / 1B)
endpoint       = os.environ["ALPHA_FOUNDRY_PROJECT_ENDPOINT"]
hub_connection = os.environ["ALPHA_FOUNDRY_CORE_CONNECTION"]  # e.g. "core-alpha"
chat_model     = os.environ["CHAT_MODEL"]                    # e.g. "gpt-4.1-mini"

# Agents reference models as {connection}/{model} — routes through the APIM connection
model_deployment = f"{hub_connection}/{chat_model}"

# Path to the CSV asset at the repository root
asset_file_path = repo_root / "assets" / "synthetic_500_quarterly_results.csv"

print(f"Endpoint  : {endpoint}")
print(f"Agent name: {AGENT_NAME}")
print(f"Model     : {model_deployment}")
print(f"Asset     : {asset_file_path}")
assert asset_file_path.exists(), f"CSV not found at {asset_file_path}"

Endpoint  : https://aif-spoke-alpha-c2676f.services.ai.azure.com/api/projects/project-alpha-c2676f
Agent name: code-interpreter-agent
Model     : core-alpha/gpt-4.1-mini
Asset     : /home/jp/development/corticalstack/foundry-nextgen/assets/synthetic_500_quarterly_results.csv


## 2. Configure authentication

`DefaultAzureCredential` resolves credentials automatically using the az CLI login, VS Code
sign-in, managed identity, or environment variables — no manual token management required.

In [2]:
credential = DefaultAzureCredential()

## 3. Create the project client

`AIProjectClient` is the main entry point for the Foundry Agent Service.

In [3]:
project_client = AIProjectClient(
    endpoint=endpoint,
    credential=credential,
)

openai_client = project_client.get_openai_client()

## 4. Upload the CSV file

The file is uploaded with `purpose="assistants"` so the Code Interpreter tool can access it
during agent execution. The returned `file.id` is passed to the agent definition in the next
step.

In [4]:
with open(asset_file_path, "rb") as f:
    uploaded_file = openai_client.files.create(purpose="assistants", file=f)

print(f"File uploaded (id: {uploaded_file.id})")

File uploaded (id: assistant-VZXMg8krZVedzZN5TPSavw)


## 5. Create an agent with Code Interpreter

`CodeInterpreterTool` enables sandboxed Python execution. `AutoCodeInterpreterToolParam`
configures an automatically managed container with the uploaded file pre-loaded.

`create_version` is idempotent — it creates a new version only when the `PromptAgentDefinition`
differs from the last stored version, making it safe to re-run while iterating.

In [5]:
agent = project_client.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model=model_deployment,
        instructions="You are a helpful data analyst assistant.",
        tools=[
            CodeInterpreterTool(
                container=AutoCodeInterpreterToolParam(file_ids=[uploaded_file.id])
            )
        ],
    ),
    description="Agent that uses Code Interpreter for data analysis and chart generation.",
)

print(f"Agent created (id: {agent.id}, name: {agent.name}, version: {agent.version})")

Agent created (id: code-interpreter-agent:1, name: code-interpreter-agent, version: 1)


## 6. Create a conversation

A conversation groups related requests into a single session. Passing `conversation.id` on
subsequent calls lets the agent maintain context across multiple turns.

In [6]:
conversation = openai_client.conversations.create()

print(f"Created conversation (id: {conversation.id})")

Created conversation (id: conv_3e82d9b97b3ff7fb005PixksNAsEu4DQpGkUIWxdd7OxFCZ0qe)


## 7. Send a prompt and generate a chart

The agent receives the prompt, writes Python code to load the CSV, filters the
TRANSPORTATION sector rows, and generates a bar chart. Setting `type` to `agent_reference`
in `extra_body` routes the request through the named agent.

In [7]:
response = openai_client.responses.create(
    conversation=conversation.id,
    input=(
        "Using the uploaded CSV file, create a bar chart showing the total operating profit "
        "per quarter for the TRANSPORTATION sector. Save the chart as a PNG file."
    ),
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
)

print(f"Response completed (id: {response.id})")

Response completed (id: resp_3e82d9b97b3ff7fb006a007aa3ef888190ae8207d54476869b)


## 8. Extract the generated file and download it

When Code Interpreter creates a file, the response contains a `container_file_citation`
annotation on the last message. Extract the `file_id` and `container_id` from that
annotation, then use `openai_client.containers.files.content.retrieve` to download the bytes.

In [8]:
file_id      = ""
filename     = ""
container_id = ""

# The last output item is the assistant message; its last content item holds annotations.
last_message = response.output[-1]
if last_message.type == "message":
    text_content = last_message.content[-1]
    if text_content.type == "output_text" and text_content.annotations:
        annotation = text_content.annotations[-1]
        if annotation.type == "container_file_citation":
            file_id      = annotation.file_id
            filename     = annotation.filename
            container_id = annotation.container_id
            print(f"Found generated file: {filename} (id: {file_id})")

if file_id and filename:
    safe_filename = Path(filename).name  # strip any path components
    file_content  = openai_client.containers.files.content.retrieve(
        file_id=file_id,
        container_id=container_id,
    )
    with open(safe_filename, "wb") as f:
        f.write(file_content.read())
    print(f"Downloaded: {safe_filename}")
else:
    print("No file annotation found in the response.")
    print("Response text:", response.output_text)

Found generated file: transportation_operating_profit_per_quarter.png (id: cfile_6a007ab1e87881909e6cf966183c004f)
Downloaded: transportation_operating_profit_per_quarter.png


## 10. Clean up *(optional)*

Uncomment the lines below to delete the agent version and the uploaded file when you no
longer need them, to avoid ongoing storage costs.

In [9]:
# project_client.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
# print(f"Deleted agent version {agent.version}")

# openai_client.files.delete(uploaded_file.id)
# print(f"Deleted uploaded file {uploaded_file.id}")